# Introduction to Agentic AI
## Introduction and Environment Setup

---

### Learning Objectives
1. Set up a **uv**-based Python 3.11 virtual environment (Windows 11 + CMD)
2. Connect the free cloud **NVIDIA build** (`deepseek-ai/deepseek-v4.1-flash`) as the default LLM, and switch to **OpenRouter** with a single line
3. Understand the difference between an LLM and an AI Agent
4. Understand the 4 core capabilities of an Agent (Tool Use / Memory / Planning / Reasoning)
5. Understand the ReAct pattern and practice the basics

> 📦 **Environment installation and run commands** are documented separately in [`env_guides/M01_1_intro_eng.md`](env_guides/M01_1_intro_eng.md).
> Repeated/shared code has been factored out into the [`agentic_lib/`](agentic_lib) library.

---
## 1. Setting Up the uv-Based Environment (Windows 11 + CMD)

### What is uv?
`uv` is an **ultra-fast Python package manager** built by Astral.
It unifies the roles of the existing `pip` + `venv` + `pyenv` into a single tool, and because it is written in Rust it is **10–100x faster** than pip.

```
pip + venv + pyenv  →  uv  (All-in-one, ultra-fast)
```

> This course uses **Windows 11 + CMD (`cmd.exe`) + Python 3.11** as its fixed target.
> All shell commands are written in CMD syntax.

### Step 1: Install uv (once — only the uv install script uses PowerShell)
```bat
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"
```

### Step 2: Create the virtual environment and install packages (from the project root)
```bat
uv venv --python 3.11
uv sync
```

### Step 3: Register the Jupyter kernel
```bat
uv run python -m ipykernel install --user --name=agentic-ai-venv --display-name "Agentic AI (uv)"
```

> Select `Agentic AI (uv)` as this notebook's **kernel** before running it.

In [1]:
# Check the uv version
import subprocess, sys

def check_tool(cmd: list, name: str):
    try:
        result = subprocess.run(cmd, capture_output=True, text=True)
        version = result.stdout.strip() or result.stderr.strip()
        print(f"✓ {name}: {version}")
        return True
    except FileNotFoundError:
        print(f"✗ {name}: not installed")
        return False

check_tool(["uv", "--version"], "uv")
check_tool([sys.executable, "--version"], "Python")
print(f"\nVirtual environment path: {sys.prefix}")
print(f"uv environment: {'✓ uv venv' if '.venv' in sys.prefix else '⚠ plain Python'}")

✓ uv: uv 0.11.8 (0e961dd9a 2026-04-27 x86_64-pc-windows-msvc)
✓ Python: Python 3.11.15

가상환경 경로: C:\Users\stshin\Documents\GitHub\Agentic AI Tutorial\.venv
uv 환경 여부: ✓ uv venv


---
## 2. Installing Packages (uv)

### pyproject.toml approach (recommended)
```toml
# pyproject.toml
[project]
name = "agentic-ai-tutorial"
version = "0.1.0"
requires-python = ">=3.11,<3.12"
dependencies = [
    "langchain>=0.3.0",
    "langchain-openai>=0.2.0",       # Used to connect to OpenAI-compatible APIs such as OpenRouter
    "langchain-nvidia-ai-endpoints", # NVIDIA build (ChatNVIDIA) — default LLM
    "langchain-google-genai>=2.0.0", # (Optional) Google Gemini
    "langchain-anthropic>=0.3.0",
    "langgraph>=0.2.0",
    "python-dotenv>=1.0.0",
    "ipykernel>=6.0.0",
]
```

```bat
REM Install everything at once from pyproject.toml
uv sync
```

### Or install directly (cell below)

In [2]:
# Install packages — uses the shared helper utils.uv_install() (uv first, falls back to pip on failure)
import sys, os
sys.path.insert(0, os.path.abspath(''))  # Add notebooks/ to the import path
from utils import uv_install

# Install the packages used in this notebook (passes quickly if already installed via uv sync)
uv_install([
    'anthropic>=0.40.0', 'google-generativeai>=0.8.0', 'langchain>=0.3.0',
    'langchain-anthropic>=0.3.0', 'langchain-google-genai>=2.0.0',
    'langchain-openai>=0.2.0', 'langchain-community>=0.3.0',
    'openai>=1.50.0', 'python-dotenv>=1.0.0',
])


[uv] 설치 완료: ['anthropic>=0.40.0', 'google-generativeai>=0.8.0', 'langchain>=0.3.0', 'langchain-anthropic>=0.3.0', 'langchain-google-genai>=2.0.0', 'langchain-openai>=0.2.0', 'langchain-community>=0.3.0', 'openai>=1.50.0', 'python-dotenv>=1.0.0']


---
## 3. Choosing an LLM Provider — the default is **NVIDIA build (`deepseek-v4.1-flash`)**

The **default LLM for this course is the free cloud API NVIDIA build + `deepseek-ai/deepseek-v4.1-flash`**
(no local GPU/server needed, fast responses, supports parallel tool calls). **OpenRouter** (`openrouter/free`) is
used alongside it for comparison, and you can switch by changing **just one line**, `LLM_PROVIDER`, in `.env` (no code changes).

| # | Provider | `LLM_PROVIDER` | Free | Recommended model | Notes |
|---|--------|----------------|------|-----------|------|
| 1 | **NVIDIA build** ⭐Default | `nvidia` | ✅ Free credits | `deepseek-ai/deepseek-v4.1-flash` | Cloud, `ChatNVIDIA` (build.nvidia.com), parallel tool calls |
| 2 | **OpenRouter** ⭐Comparison | `openrouter` | ✅ Free models | `openrouter/free` | Cloud, OpenAI-compatible `/v1`, automatic routing to free models |
| 3 | Google AI Studio | `google` | ✅ Free tier | `gemini-3.1-flash-lite` | (Optional) Cloud |
| 4 | Anthropic (API/OAuth) | `anthropic` | Paid | `claude-haiku-4-5` | (Optional) Safety, long context |
| 5 | OpenAI | `openai` | Paid | `gpt-4o-mini` | (Optional) General-purpose |

> The `ollama` · `llamacpp` · `vllm` (local server) providers still remain in `utils.get_llm()`, but they are not used in this course.

### Getting free cloud API keys (required)
- **NVIDIA build**: Log in at [build.nvidia.com](https://build.nvidia.com) → model page → `Get API Key` → put `NVIDIA_API_KEY=nvapi-...` in `.env`
- **OpenRouter**: Log in at [openrouter.ai](https://openrouter.ai) → `Keys` → `Create Key` → put `OPENROUTER_API_KEY=sk-or-...` in `.env`

> All providers are used **the same way** through `utils.get_llm()` (§6-B connects to and tests the two free APIs).
> For detailed installation and troubleshooting, see [`env_guides/M01_1_intro_eng.md`](env_guides/M01_1_intro_eng.md).

In [3]:
import sys, os
sys.path.insert(0, os.path.abspath(''))  # Add the notebooks/ directory to the import path

import utils
utils.reload_env()  # Reload .env (refreshes LLM_PROVIDER, etc.) + print the current provider status

from utils import (
    uv_install, get_llm, test_llm_connection,
    LLM_PROVIDER, GOOGLE_API_KEY, ANTHROPIC_API_KEY, ANTHROPIC_OAUTH_TOKEN,
    VLLM_BASE_URL, VLLM_MODEL, OPENAI_API_KEY,
    NVIDIA_API_KEY, NVIDIA_MODEL,   # For connecting to NVIDIA build (free credits)
    print_provider_status,
)

# Repeated/shared code has been factored out into the agentic_lib library (used throughout this notebook).
from agentic_lib import bootstrap, tools, memory, planning, react
from agentic_lib.bootstrap import to_text  # Provider-agnostic response normalization (including <think> removal)

LLM 공급자: nvidia
  NVIDIA build Key: 설정됨  /  Model: meta/llama-3.1-8b-instruct


---
## 4. Authentication/Configuration (`.env`)

Copy `notebooks/.env.example` to create `notebooks/.env`, and fill in only the providers you use.

```bat
cd notebooks
copy .env.example .env
```

```ini
# notebooks/.env  — default is NVIDIA build
LLM_PROVIDER=nvidia
NVIDIA_API_KEY=nvapi-...
NVIDIA_MODEL=deepseek-ai/deepseek-v4.1-flash   # (Optional) default value

# To compare with OpenRouter, change LLM_PROVIDER above to openrouter
OPENROUTER_API_KEY=sk-or-...
OPENROUTER_MODEL=openrouter/free               # (Optional) default value — automatically selects a free model
```

> **Getting an NVIDIA API Key (free credits)**: Log in at [build.nvidia.com](https://build.nvidia.com) → model page → `Get API Key`
> → save `NVIDIA_API_KEY=nvapi-...` in `.env`, then set `LLM_PROVIDER=nvidia`
>
> **Getting an OpenRouter API Key (free models)**: [openrouter.ai](https://openrouter.ai) → `Keys` → `Create Key`
> → save `OPENROUTER_API_KEY=sk-or-...` in `.env`, then set `LLM_PROVIDER=openrouter`

In [4]:
# If a .env file exists, load_dotenv() is called automatically when utils.py is imported.
# If you want to double-check, run the line below.
import os
print('LLM_PROVIDER:', os.getenv('LLM_PROVIDER', '(not set — default: google)'))


LLM_PROVIDER: nvidia


---
## 5. Detailed Connection Settings per Provider

In [5]:
# ============================================================
# [Reference] Per-provider connections — unified in get_llm() in utils.py
# ============================================================
# get_llm() handles the settings below automatically (selected by LLM_PROVIDER in .env):
#
#   "nvidia"     → ChatNVIDIA(model="deepseek-ai/deepseek-v4.1-flash")              ⭐Default
#   "openrouter" → ChatOpenAI(base_url=OPENROUTER_BASE_URL, model="openrouter/free") ⭐Comparison
#   "google"     → ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")
#   "anthropic"  → ChatAnthropic(model="claude-haiku-4-5-20251001")
#   "openai"     → ChatOpenAI(model="gpt-4o-mini")
#
# Every provider returns a LangChain BaseChatModel, so the .invoke()/.bind_tools() interface is the same.
print("Per-provider connections are unified in utils.get_llm().")
print(f"Currently selected provider: {LLM_PROVIDER}")

공급자별 연결은 utils.get_llm() 으로 통합됩니다.
현재 선택된 공급자: nvidia


---
## 6. Unified LLM Interface (common to all providers)

In the following exercises, all providers are used the same way through a single `get_llm()` function.

In [6]:
# get_llm() is defined in utils.py and has already been imported.
# To test with other providers:
# llm_nvidia     = get_llm('nvidia')
# llm_openrouter = get_llm('openrouter')
# llm_google     = get_llm('google')
print('get_llm() is ready to use. Call llm = get_llm() or test_llm_connection().')


get_llm() 사용 준비 완료. llm = get_llm() 또는 test_llm_connection()을 호출하세요.


In [7]:
llm = test_llm_connection()

LLM 연결 성공 [nvidia]: 1+1은 2입니다.


---
## 6-B. Testing the Free Cloud LLM APIs — NVIDIA build & OpenRouter

We connect to and test each of the **two free cloud APIs** used in this course via `utils.get_llm()`,
independently of `LLM_PROVIDER` in `.env`.

| Service | Free | Get a key | Connection (shared factory) | Example model |
|---|---|---|---|---|
| **NVIDIA build** ⭐Default | ✅ Free credits | [build.nvidia.com](https://build.nvidia.com) | `utils.get_llm('nvidia')` | `deepseek-ai/deepseek-v4.1-flash` |
| **OpenRouter** | ✅ Free models | [openrouter.ai](https://openrouter.ai) | `utils.get_llm('openrouter')` | `openrouter/free` |

- Fill in `NVIDIA_API_KEY` and `OPENROUTER_API_KEY` in `.env`. For services without a key, only **key-issuing guidance** is printed, with no error.
- For both services, `utils.get_llm()` returns the appropriate connector (`ChatNVIDIA` / `ChatOpenAI`), so they are handled through the **same `.invoke()` interface**.
- For more detailed examples such as streaming comparisons, see [`M02_0_free_llm_api_eng.ipynb`](M02_0_free_llm_api_eng.ipynb).

In [8]:
# === Connecting to and testing the free cloud LLM APIs (NVIDIA build · OpenRouter) ===
# Independently of LLM_PROVIDER in .env, we call the two free cloud APIs directly via utils.get_llm().
from langchain_core.messages import HumanMessage, SystemMessage

# Install the LangChain connectors for both services (passes quickly if already installed)
uv_install(['langchain-nvidia-ai-endpoints', 'langchain-openai'])

# Key configuration status (loaded from .env)
print('NVIDIA build key:', 'set' if NVIDIA_API_KEY else 'not set — build.nvidia.com')
print('OpenRouter key  :', 'set' if utils.OPENROUTER_API_KEY else 'not set — openrouter.ai')
print('-' * 60)

# Shared test prompt
CLOUD_PROMPT = [
    SystemMessage(content='Answer concisely in English.'),
    HumanMessage(content='Explain Agentic AI in one sentence.'),
]

def test_cloud_api(provider: str, label: str, api_key: str):
    """Test connecting to and calling a single free cloud provider (prints guidance only if there is no key)."""
    if not api_key:
        print(f'[{label}] Key not set — add it to .env after obtaining one to run this test.')
        return
    try:
        # nvidia → ChatNVIDIA / openrouter → ChatOpenAI(base_url=OpenRouter) (same interface)
        model = get_llm(provider)
        resp = model.invoke(CLOUD_PROMPT)
        print(f'[{label}] {to_text(resp.content)}')   # Provider-agnostic response normalization
    except Exception as e:
        print(f'[{label}] Call failed: {type(e).__name__}: {e}')

# Connection test for each service (always runs, regardless of LLM_PROVIDER in .env)
test_cloud_api('nvidia', f'NVIDIA {NVIDIA_MODEL}', NVIDIA_API_KEY)
test_cloud_api('openrouter', f'OpenRouter {utils.OPENROUTER_MODEL}', utils.OPENROUTER_API_KEY)

[uv] 설치 완료: ['langchain-google-genai', 'langchain-nvidia-ai-endpoints']
Google AI Studio 키: 설정됨
NVIDIA build 키   : 설정됨
------------------------------------------------------------


[Google Gemini] 에이전틱 AI는 단순히 명령을 수행하는 것을 넘어, 스스로 목표를 설정하고 계획을 세워 자율적으로 문제를 해결하는 지능형 시스템입니다.


[NVIDIA meta/llama-3.1-8b-instruct] 에이전틱(Agentic) AI는 사용자의 의도와 목표를 이해하고 이를 달성하기 위한 행동을 취하는 인공지능을 말합니다.


---
## 7. LLM vs AI Agent: What's the Difference?

### Plain LLM
- **Role:** Generates text output for input text
- **Limitations:** Answers based on training data, no real-time information, cannot use external tools
- **Analogy:** Even the most knowledgeable professor, only able to talk without internet access

### AI Agent
- **Role:** Plans on its own, uses tools, and evaluates results to achieve a goal
- **Strengths:** Real-time information retrieval, code execution, external API calls, multi-step planning
- **Analogy:** Like a personal assistant who actually searches the internet, coordinates schedules, and even writes documents

```
[LLM approach]
User question → LLM → Text answer

[Agent approach]
User goal → Agent ┌→ Make a plan
                  ├→ Select and execute tools
                  ├→ Observe results
                  ├→ Re-plan (if needed)
                  └→ Final answer
```

---
## 8. The 4 Core Capabilities of an Agent

| Capability | Analogy | Role |
|------|------|------|
| **Tool Use** | Hands | External API calls, web search, code execution |
| **Memory** | Memory | Storing and using past conversations and long-term knowledge |
| **Planning** | Strategy | Breaking complex problems down into steps |
| **Reasoning** | Intelligence | Logical thinking and decision-making with real-time feedback |

In [9]:
# === The 4 core capabilities of an Agent — demo with a real LLM ===
# Shared tools come from agentic_lib.tools, and responses are printed cleanly with to_text().
# (content comes as list/str depending on the provider, and thinking models mix in <think>, so we unify with to_text)
from langchain_core.messages import HumanMessage, ToolMessage, SystemMessage

if llm is None:
    print('LLM is not connected.')
else:
    # ── 1. Tool Use ──────────────────────────────────────────────────────
    print('━' * 60)
    print('1. Tool Use: the LLM calls external tools directly')
    print('━' * 60)

    # calculator/get_current_time are provided by the library (duplicate definitions removed)
    available_tools = [tools.calculator, tools.get_current_time]
    # bootstrap.bind_tools: forces parallel_tool_calls=False when switched to a single-tool-only model
    llm_with_tools = bootstrap.bind_tools(llm, available_tools)

    # Tool name → tool object mapping (registry). Finds and executes the right tool by the name the LLM chose.
    tool_registry = {t.name: t for t in available_tools}

    question = 'What is 2 to the power of 10?'
    print(f'[Question] {question}')
    resp = llm_with_tools.invoke([HumanMessage(content=question)])
    # On a single-tool server, trim tool_calls to the first one so history holds only one tool (otherwise keep original)
    resp = bootstrap.cap_tool_calls(resp)

    if resp.tool_calls:
        tc = resp.tool_calls[0]
        tool_name = tc['name']
        print(f'[LLM decision] Tool selected: {tool_name}({tc["args"]})')
        # Instead of hard-coding, look up the right tool by name and execute it
        selected_tool = tool_registry.get(tool_name)
        if selected_tool is None:
            tool_result = 'Unknown tool: ' + tool_name
        else:
            tool_result = selected_tool.invoke(tc['args'])
        print(f'[Tool execution] {tool_result}')
        final = llm_with_tools.invoke([
            HumanMessage(content=question), resp,
            ToolMessage(content=str(tool_result), tool_call_id=tc['id'])
        ])
        print(f'[Final answer] {to_text(final.content)}')  # Normalize list/think with to_text
    else:
        print(f'[LLM answer (no tool used)] {to_text(resp.content)}')

    # ── 2. Memory ─────────────────────────────────────────────────────────
    print('\n' + '━' * 60)
    print('2. Memory: remembers previous conversation')
    print('━' * 60)

    history = [SystemMessage(content='You are an AI assistant that remembers user information.')]
    for user_input in ['My name is Chulsoo Kim and I live in Seoul.', 'What did I say my name was and where I live?']:
        history.append(HumanMessage(content=user_input))
        print(f'[User] {user_input}')
        reply = llm.invoke(history)
        history.append(reply)
        print(f'[Agent] {to_text(reply.content)[:120]}')

    # ── 3. Planning ───────────────────────────────────────────────────────
    print('\n' + '━' * 60)
    print('3. Planning: the LLM breaks a complex task into steps')
    print('━' * 60)

    plan_resp = llm.invoke([
        SystemMessage(content='Break the request down into numbered, concrete execution steps. One line per step.'),
        HumanMessage(content='Make a plan for writing an AI trend analysis report.')
    ])
    print(f'[Plan]\n{to_text(plan_resp.content)}')

    # ── 4. Reasoning ──────────────────────────────────────────────────────
    print('\n' + '━' * 60)
    print('4. Reasoning: step-by-step logical reasoning (Chain of Thought)')
    print('━' * 60)

    cot_resp = llm.invoke([
        SystemMessage(content=(
            'Reason through the problem logically, step by step. '
            'Explain each step clearly in the form "Step N:" and then give the final answer.'
        )),
        HumanMessage(content=(
            'There are 24 apples, and 6 people share them equally. '
            'If each person then eats half of their share, how many apples are left in total?'
        ))
    ])
    print(f'[Reasoning]\n{to_text(cot_resp.content)}')

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. Tool Use: LLM 이 외부 도구를 직접 호출합니다
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[질문] 2의 10제곱이 얼마야?


[LLM 결정] 도구 선택: calculator({'expression': '2 ** 10'})
[도구 실행] 2 ** 10 = 1024


[최종 답변] 2의 10제곱은 1024입니다.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
2. Memory: 이전 대화를 기억합니다
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[사용자] 제 이름은 김철수이고 서울에 살아요.


[에이전트] 안녕하세요 김철수님! 서울에 살고 계신 것 같아요. 어떻게 지내시나요?
[사용자] 제 이름이 뭐고 어디 산다고 했죠?


[에이전트] 네, 김철수님이라고 하셨고, 서울에 살고 계신다고 말씀하셨습니다.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
3. Planning: LLM 이 복잡한 작업을 단계별로 분해합니다
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


[계획]
1. **보고서 목적과 범위 정의**: AI 트렌드 분석 보고서의 목적, 범위, 대상audience를 명확히 정의합니다.
2. **AI 트렌드 수집**: AI 트렌드 분석을 위한 데이터 수집을 위해 관련 기사, 연구-paper, 보고서, 컨퍼런스 자료 등 다양한 소스를 수집합니다.
3. **트렌드 분류**: 수집한 데이터를 AI 트렌드의 유형(예: 자연어 처리, computer vision, 강화 학습 등)으로 분류합니다.
4. **트렌드 분석**: 각 트렌드의 특징, 장점, 단점, 예상 영향, 예상 시기 등을 분석합니다.
5. **트렌드 예측**: 분석 결과를 기반으로 AI 트렌드의 미래 예측을 합니다.
6. **보고서 구조 설계**: 보고서의 구조를 설계합니다. 일반적으로 Executive Summary, Introduction, Methodology, Analysis, Conclusion, Recommendation 등으로 구성됩니다.
7. **Executive Summary 작성**: 보고서의 주요 내용을 요약하여 Executive Summary를 작성합니다.
8. **Introduction 작성**: 보고서의 목적, 범위, 대상audience를 설명하여 Introduction을 작성합니다.
9. **Methodology 작성**: 데이터 수집, 분류, 분석 방법을 설명하여 Methodology를 작성합니다.
10. **Analysis 작성**: 각 트렌드의 특징, 장점, 단점, 예상 영향, 예상 시기 등을 분석하여 Analysis를 작성합니다.
11. **Conclusion 작성**: 분석 결과를 요약하여 Conclusion을 작성합니다.
12. **Recommendation 작성**: 분석 결과를 기반으로 AI 트렌드에 대한 추천을 작성합니다.
13. **보고서 작성**: 각 섹션을 작성하여 보고서를 완성합니다.
14. **보고서 리뷰**: 보고서를 리뷰하여 오류를 수정하고 내용을 보완합니다.
15. **보고서 최종화**: 보고서를 최종화하

[추론]
단계 1: 사과가 24개 있고 6명이 똑같이 나눠 가진 경우, 각 사람의 몫은 24 / 6 = 4개입니다.

단계 2: 각 사람의 몫에서 절반을 먹었다면, 각 사람의 남은 사과는 4 / 2 = 2개입니다.

단계 3: 6명이 모두 남은 사과를 가지고 있으므로, 총 남은 사과는 6 * 2 = 12개입니다.

최종 답: 남은 사과는 총 12개입니다.


---
## 9. The ReAct Pattern (Reason + Act)

ReAct is the core thinking loop of an agent:

```
Goal input
  ↓
Thought: analyze the current situation, decide the next action
  ↓
Action: select and execute a tool
  ↓
Observation: check the tool's result
  ↓
(repeat or finish)
  ↓
Final Answer: present the final answer
```

**Paper:** Yao et al., "ReAct: Synergizing Reasoning and Acting in Language Models" (ICLR 2023)

In [10]:
# Implementing the ReAct pattern by hand (to understand the logic, without an LLM) — uses agentic_lib.react
# SimpleReActAgent / rule-based tools (PLAIN_TOOLS) are factored out into the library.
agent = react.SimpleReActAgent()          # Default tools = react.PLAIN_TOOLS
TOOLS = react.PLAIN_TOOLS                 # Reused in the Planning demo below

print('=' * 60); print('Test 1: Current time'); print('=' * 60)
# Note: SimpleReActAgent picks tools by matching Korean keywords (for "time"), so this query
# intentionally stays in Korean. It means "What time is it now?"
agent.run('지금 몇 시야?')
print(); print('=' * 60); print('Test 2: Math calculation'); print('=' * 60)
agent.run('Calculate 2 ** 10')

테스트 1: 현재 시간
[목표] 지금 몇 시야?
--------------------------------------------------
[Thought 1] 사용할 도구: get_current_time
[Action 1] get_current_time('')
[Observation 1] 2026년 07월 25일 15:27:09

[Final Answer] 2026년 07월 25일 15:27:09

테스트 2: 수학 계산
[목표] 2 ** 10 계산해줘
--------------------------------------------------
[Thought 1] 사용할 도구: calculator
[Action 1] calculator('2 ** 10')
[Observation 1] 2 ** 10 = 1024

[Final Answer] 2 ** 10 = 1024


'2 ** 10 = 1024'

---
## 10. Implementing a Real ReAct Agent with the Selected LLM

We use each provider's Tool Calling API through a unified interface.

In [11]:
from langchain.agents import create_agent

# LangChain tools use the shared tools from agentic_lib.tools (duplicate definitions removed)
lc_tools = [tools.calculator, tools.get_current_time, tools.search_web]

system_prompt = (
    "You are a capable AI agent that uses tools. "
    "Analyze the user's request, choose the appropriate tools, and provide an accurate answer. "
    "Always answer in English."
)

def run_react_agent(question: str):
    """Run a ReAct agent (create_agent) with the selected LLM provider."""
    if llm is None:
        print("LLM is not connected. Run section 6 first.")
        return
    try:
        agent = create_agent(llm, lc_tools, system_prompt=system_prompt)
        result = agent.invoke({"messages": [("human", question)]})
        print(f"\nFinal answer: {to_text(result['messages'][-1].content)}")
        return result
    except Exception as e:
        print(f"Agent error: {type(e).__name__}: {e}")

print(f"=== ReAct Agent ({LLM_PROVIDER}) ===")
run_react_agent("Tell me what time it is now, and also calculate 2 to the power of 10")

=== ReAct Agent (nvidia) ===


에이전트 오류: Exception: [500] {'message': 'Failed to generate completions: Failed to apply prompt template: invalid operation: This model only supports single tool-calls at once! (in tool_use:95)', 'type': 'Internal Server Error', 'code': 500}
{'error': {'message': 'Failed to generate completions: Failed to apply prompt template: invalid operation: This model only supports single tool-calls at once! (in tool_use:95)', 'type': 'Internal Server Error', 'code': 500}, 'status': 500}


---
## 11. Comparing Direct Calls per Provider

We also look at how to use each provider's SDK directly, without LangChain.

In [12]:
def call_llm_direct(prompt: str, provider: str = None) -> str:
    """
    Calls each provider's SDK directly (without LangChain).
    Reference code for understanding the differences between the SDKs.
    (NVIDIA build and OpenRouter are OpenAI-compatible APIs, so they are called with the OpenAI SDK by changing only base_url)
    """
    p = provider or LLM_PROVIDER

    # ── Google AI Studio ────────────────────────────────────
    if p == "google":
        from google import genai
        client = genai.Client(api_key=GOOGLE_API_KEY)
        response = client.models.generate_content(
            model="gemini-3.1-flash-lite",
            contents=prompt,
        )
        return response.text

    # ── Anthropic API Key ────────────────────────────────────
    elif p == "anthropic":
        import anthropic
        client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=256,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.content[0].text

    # ── Anthropic OAuth ──────────────────────────────────────
    elif p == "anthropic_oauth":
        import anthropic
        client = anthropic.Anthropic(auth_token=ANTHROPIC_OAUTH_TOKEN)
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=256,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.content[0].text

    # ── vLLM (OpenAI-compatible) ─────────────────────────────
    elif p == "vllm":
        from openai import OpenAI
        client = OpenAI(base_url=VLLM_BASE_URL, api_key="EMPTY")
        response = client.chat.completions.create(
            model=VLLM_MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=256
        )
        return response.choices[0].message.content

    # ── OpenAI ──────────────────────────────────────────────
    elif p == "openai":
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=256
        )
        return response.choices[0].message.content

    # ── NVIDIA build / OpenRouter (cloud, OpenAI-compatible) ─
    elif p in ("nvidia", "openrouter"):
        from openai import OpenAI
        # Both services provide an OpenAI-compatible /v1, so we call them by changing only base_url, key, and model
        if p == "nvidia":
            base_url = utils.NVIDIA_BASE_URL or "https://integrate.api.nvidia.com/v1"
            api_key, model = utils.NVIDIA_API_KEY, utils.NVIDIA_MODEL
        else:
            base_url = utils.OPENROUTER_BASE_URL
            api_key, model = utils.OPENROUTER_API_KEY, utils.OPENROUTER_MODEL
        client = OpenAI(base_url=base_url, api_key=api_key)
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=512,  # Reasoning (thinking) models spend tokens on reasoning, so be generous
        )
        # If a thinking model's <think> block is mixed in, remove it and return a clean string
        return bootstrap.strip_think(response.choices[0].message.content or "")

    return f"Unsupported provider: {p}"


# Direct call test
test_prompt = "Please explain Agentic AI in one sentence."
try:
    answer = call_llm_direct(test_prompt)
    print(f"[{LLM_PROVIDER}] Direct call result:")
    print(f"  {answer}")
except Exception as e:
    print(f"Call failed: {type(e).__name__}: {e}")

[nvidia] 직접 호출 결과:
  지원하지 않는 공급자: nvidia


---
## 12. Hands-on: The Memory Concept

- **Short-term memory:** Maintains the context of the current conversation
- **Long-term memory:** Stored permanently in external storage

In [13]:
# Short-/long-term memory — uses agentic_lib.memory (class implementations live in the library)
short = memory.ConversationMemory(max_messages=5)
long  = memory.SimpleVectorMemory()

short.add("user", "My name is Chulsoo Kim.")
short.add("assistant", "Hello, Chulsoo Kim!")
short.add("user", "I am an AI developer.")

print("=== Short-term memory (sliding window) ===")
for m in short.get():
    print(f"  [{m['role']}] {m['content']}")

long.save("user_profile", "Name: Chulsoo Kim, Occupation: AI developer")
long.save("project_A", "Implementing a LangGraph-based agent system")

print("\n=== Long-term memory search ('Chulsoo Kim') ===")
for r in long.search("Chulsoo Kim"):
    print(f"  [{r['key']}] {r['value']}")

=== 단기 기억 (슬라이딩 윈도우) ===
  [user] 제 이름은 김철수입니다.
  [assistant] 안녕하세요, 김철수님!
  [user] 저는 AI 개발자입니다.
[저장] '사용자_프로필'
[저장] '프로젝트_A'

=== 장기 기억 검색 ('김철수') ===
  [사용자_프로필] 이름: 김철수, 직업: AI 개발자


---
## 13. Hands-on: The Planning Concept

In [14]:
# Task planning and execution — uses agentic_lib.planning (Task/TaskStatus/TaskPlanner live in the library)
planner = planning.TaskPlanner()
t1 = planner.add("Search AI trends", tool="search_web")
t2 = planner.add("Check the current date", tool="get_current_time")
t3 = planner.add("Design the report structure", depends_on=[t1.id, t2.id])
t4 = planner.add("Write the report draft", depends_on=[t3.id])
t5 = planner.add("Review and finish",    depends_on=[t4.id])

planner.show()
planner.execute(TOOLS)   # TOOLS = react.PLAIN_TOOLS (defined in an earlier cell)
print()
planner.show()

=== 실행 계획 ===
  [대기  ] Task 1: AI 트렌드 검색
  [대기  ] Task 2: 현재 날짜 확인
  [대기  ] Task 3: 보고서 구조 설계 (의존: [1, 2])
  [대기  ] Task 4: 보고서 초안 작성 (의존: [3])
  [대기  ] Task 5: 검토 및 완료 (의존: [4])

=== 실행 ===
  Task 1: AI 트렌드 검색
    → 'AI 트렌드 검색' 검색 결과: 관련 정보 3건 발견 (시뮬레이션)
  Task 2: 현재 날짜 확인
    → 2026년 07월 25일 15:27:12
  Task 3: 보고서 구조 설계
    → 완료
  Task 4: 보고서 초안 작성
    → 완료
  Task 5: 검토 및 완료
    → 완료

=== 실행 계획 ===
  [완료  ] Task 1: AI 트렌드 검색
  [완료  ] Task 2: 현재 날짜 확인
  [완료  ] Task 3: 보고서 구조 설계 (의존: [1, 2])
  [완료  ] Task 4: 보고서 초안 작성 (의존: [3])
  [완료  ] Task 5: 검토 및 완료 (의존: [4])


---
## 14. Summary and Preview of the Next Module

### Key Takeaways from This Session

| Topic | Key points |
|------|----------|
| **uv** | A package manager 10–100x faster than pip. `uv venv --python 3.11` + `uv sync` |
| **Default LLM** | **NVIDIA build + `deepseek-v4.1-flash`** (free credits, no local server needed). `openrouter` is a one-line switch for comparison |
| **Free cloud APIs** | **NVIDIA build (`ChatNVIDIA`)** · **OpenRouter (`ChatOpenAI`)** — connect and test with `utils.get_llm('nvidia'/'openrouter')` (§6-B) |
| **LLM vs Agent** | LLM = text generator / Agent = autonomous execution system |
| **4 capabilities** | Tool Use (hands) · Memory (memory) · Planning (strategy) · Reasoning (intelligence) |
| **ReAct** | Repeated Thought → Action → Observation loop |
| **Library** | Repeated code is factored out into `agentic_lib` (tools · memory · planning · react · bootstrap) |

### LLM Provider Selection Guide
```
Free cloud (default, recommended)      → NVIDIA build        (LLM_PROVIDER=nvidia)
Auto-routing to free models (compare)  → OpenRouter          (LLM_PROVIDER=openrouter)
(Optional) Google free tier            → Google Gemini       (LLM_PROVIDER=google)
(Optional) Safety, long context        → Anthropic Claude
```

### Next Week: Module 1 - Connectivity (Connectivity & Skills)
- **MCP (Model Context Protocol):** A standard for connecting agents to external services
- **A2A (Agent-to-Agent):** An architecture for collaboration between agents
- **OpenAI-compatible protocol:** Connect NVIDIA build / OpenRouter with the same code and integrate tools

---
### References
- uv official documentation: https://docs.astral.sh/uv/
- NVIDIA build: https://build.nvidia.com  ·  LangChain connector: `langchain-nvidia-ai-endpoints`
- OpenRouter: https://openrouter.ai  ·  Free models: https://openrouter.ai/models?q=free
- Anthropic API documentation: https://docs.anthropic.com
- ReAct paper: https://arxiv.org/abs/2210.03629